In [3]:
# Imports principales
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
#import easyocr
import pandas as pd
from datetime import datetime
import glob
import re

# Configurar matplotlib
%matplotlib inline

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="um9ufCqGkzHq1f3P1YQr")
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version = project.version(12)
dataset = version.download("yolov11")
                

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 36.7 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 20.4 MB/s eta 0:00:00m eta 0:00:01
  Attempting uninstall: opencv-python-headless━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [filetype]
    Found existing installation: opencv-python-headless 4.12.0.88━━━━━━━━━━━━━━━━━━━━━━ 1/5 [filetype]
    Uninstalling opencv-python-headless-4.12.0.88:╺━━━━━━━━━━━━━━━ 3/5 [opencv-python-headless]
      Successfully uninstalled opencv-python-headless-4.12.0.880m━━━━━━━━━━━━━━━ 3/5 [opencv-python-headless]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [roboflow]━━ 4/5 [roboflow]thon-headless]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
label-studio-sdk 2.0.16 requires opencv-python-headless<5.0.0,>=4.12


Extracting Dataset Version Zip to License-Plate-Recognition-12 in yolov11:: 100%|██████████| 20262/20262 [00:01<00:00, 15951.32it/s]


In [ ]:
# Si quieres entrenar tu propio modelo
model = YOLO("yolo11n.pt")  # Modelo nano (más rápido)

# Entrenar
results = model.train(
    data='License-Plate-Recognition-12/data.yaml',
    epochs=15,
    imgsz=640,
    batch=16,
    name='license_plate_detector'
)

# Guardar mejor modelo
!cp /content/runs/detect/license_plate_detector/weights/best.pt /content/best_license_plate.pt

New https://pypi.org/project/ultralytics/8.4.15 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.235 🚀 Python-3.13.5 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 12044MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=License-Plate-Recognition-12/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=license_plate

In [ ]:
def detectar_matricula(image_path, model, conf_threshold=0.5):
    """
    Detecta la matrícula en una imagen usando YOLO
    
    Args:
        image_path: Ruta a la imagen
        model: Modelo YOLO cargado
        conf_threshold: Umbral de confianza mínimo
    
    Returns:
        tuple: (imagen_original, coordenadas_matricula, confianza)
    """
    # Leer imagen
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Hacer predicción
    results = model(image_path, conf=conf_threshold)
    
    # Obtener detecciones
    if len(results[0].boxes) == 0:
        print("❌ No se detectó ninguna matrícula")
        return img_rgb, None, 0
    
    # Obtener la detección con mayor confianza
    boxes = results[0].boxes
    confidences = boxes.conf.cpu().numpy()
    best_idx = np.argmax(confidences)
    
    # Coordenadas del bounding box (x1, y1, x2, y2)
    bbox = boxes.xyxy[best_idx].cpu().numpy().astype(int)
    conf = confidences[best_idx]
    
    print(f"✅ Matrícula detectada con confianza: {conf:.2%}")
    print(f"   Coordenadas: {bbox}")
    
    return img_rgb, bbox, conf


# Ejemplo de uso
img_path = 'License-Plate-Recognition-12/test/images/0010f4c10f7ab07e_jpg.rf.1844f6dde3b97ed1c762db933bbacaf3.jpg'
imagen, coordenadas, confianza = detectar_matricula(img_path, model)

# Visualizar detección
if coordenadas is not None:
    x1, y1, x2, y2 = coordenadas
    img_con_box = imagen.copy()
    cv2.rectangle(img_con_box, (x1, y1), (x2, y2), (0, 255, 0), 3)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(img_con_box)
    plt.title(f'Matrícula Detectada (Conf: {confianza:.2%})')
    plt.axis('off')
    plt.show()